In [0]:
FEATURE_TABLE_NAME = "workspace.marketing_campaign.gold_customer_features"

features_df = spark.table(FEATURE_TABLE_NAME)

print(f"Feature table: {FEATURE_TABLE_NAME}")
print(f"Rows: {features_df.count()}")
print(f"Columns: {len(features_df.columns)}")

display(features_df.limit(10))

In [0]:
model_df = features_df.select(
    "education",
    "marital_status",
    "income",
    "customer_age",
    "customer_tenure_days",
    "has_children",
    "recency",
    "total_spend",
    "total_purchases",
    "numwebvisitsmonth",
    "acceptedcmp1",
    "acceptedcmp2",
    "acceptedcmp3",
    "acceptedcmp4",
    "acceptedcmp5",
    "accepted_previous_campaign",
    "complain",
    "response"
).dropna()

print(f"Model rows after dropping nulls: {model_df.count()}")
display(model_df.limit(10))

In [0]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_df.count()}")
print(f"Test rows: {test_df.count()}")

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

categorical_columns = ["education", "marital_status"]

numeric_columns = [
    "income",
    "customer_age",
    "customer_tenure_days",
    "has_children",
    "recency",
    "total_spend",
    "total_purchases",
    "numwebvisitsmonth",
    "acceptedcmp1",
    "acceptedcmp2",
    "acceptedcmp3",
    "acceptedcmp4",
    "acceptedcmp5",
    "accepted_previous_campaign",
    "complain"
]

indexers = [
    StringIndexer(
        inputCol=column_name,
        outputCol=f"{column_name}_index",
        handleInvalid="keep"
    )
    for column_name in categorical_columns
]

encoders = [
    OneHotEncoder(
        inputCol=f"{column_name}_index",
        outputCol=f"{column_name}_encoded"
    )
    for column_name in categorical_columns
]

assembler = VectorAssembler(
    inputCols=numeric_columns + [f"{column_name}_encoded" for column_name in categorical_columns],
    outputCol="features"
)

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="response",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=20
)

pipeline = Pipeline(
    stages=indexers + encoders + [assembler, logistic_regression]
)

In [0]:
import mlflow

mlflow.set_experiment("/Users/ishita.k2208@gmail.com/marketing_campaign_response")

with mlflow.start_run(run_name="logistic_regression_baseline"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("feature_table", FEATURE_TABLE_NAME)
    mlflow.log_param("train_rows", train_df.count())
    mlflow.log_param("test_rows", test_df.count())

    model = pipeline.fit(train_df)

    predictions_df = model.transform(test_df)

print("Model training complete.")
print("Model parameters logged to MLflow.")

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

predictions_df = model.transform(test_df)

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="response",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="response",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="response",
    predictionCol="prediction",
    metricName="f1"
)

roc_auc = auc_evaluator.evaluate(predictions_df)
accuracy = accuracy_evaluator.evaluate(predictions_df)
f1_score = f1_evaluator.evaluate(predictions_df)

print(f"ROC AUC: {roc_auc}")
print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1_score}")

In [0]:
with mlflow.start_run(run_name="logistic_regression_metrics"):
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1_score)

print("Metrics logged to MLflow.")

In [0]:
display(
    predictions_df.select(
        "education",
        "marital_status",
        "income",
        "customer_age",
        "total_spend",
        "total_purchases",
        "response",
        "prediction",
        "probability"
    ).limit(20)
)